# Backfill Completo — Reset e Regeneração da História do Projeto

Executa o backfill único planejado no ADR-010: reset completo de todas as camadas (Bronze, Silver, Gold/observability, pipeline_runs, Landing Zone, checkpoints), seguido de geração de 60 dias de histórico simulado (16/06/2026 a 14/08/2026).

Motivado por um erro real encontrado no primeiro teste do Job via Asset Bundles: checkpoint do Autoloader referenciando uma partição da Landing Zone já removida por testes manuais anteriores — exatamente o cenário que o ADR-010 já previa como razão para nunca tentar manter continuidade de dado de desenvolvimento.

Executado em 2 etapas: piloto de 5 dias primeiro, depois os 60 dias completos.

Referências: ADR-009, ADR-010, ADR-011.

In [0]:
# Reset completo — todas as camadas
todas_as_tabelas_evento = [
    "erp_lotes_producao", "erp_posicoes_estoque", "erp_notas_expedicao",
    "crm_pedidos", "crm_itens_pedido", "crm_atendimento",
    "tms_remessas", "tms_leituras_temperatura", "tms_comprovantes_entrega",
    "financeiro_faturas", "financeiro_contas_receber",
]
todas_as_tabelas_seed = [
    "erp_produtos", "crm_representantes", "crm_clientes",
    "tms_veiculos", "tms_rotas", "financeiro_centros_custo",
]
todas_as_tabelas_gold = [
    "gold_reconciliacao_financeira", "gold_otif", "gold_qualidade_producao",
    "observability_cadeia_fria", "observability_qualidade_sku", "observability_estoque_negativo",
]
sistemas = ["erp", "crm", "tms", "financeiro"]

# 1. Bronze e Silver das tabelas de evento
for tabela in todas_as_tabelas_evento:
    spark.sql(f"DROP TABLE IF EXISTS poc_pulse_observability.bronze.{tabela}")
    spark.sql(f"DROP TABLE IF EXISTS poc_pulse_observability.silver.{tabela}")
    dbutils.fs.rm(f"/Volumes/poc_pulse_observability/landing/raw/_autoloader_checkpoint/{tabela}", recurse=True)
    dbutils.fs.rm(f"/Volumes/poc_pulse_observability/landing/raw/_autoloader_schema/{tabela}", recurse=True)

# 2. Bronze e Silver dos seeds
for tabela in todas_as_tabelas_seed:
    spark.sql(f"DROP TABLE IF EXISTS poc_pulse_observability.bronze.{tabela}")
    spark.sql(f"DROP TABLE IF EXISTS poc_pulse_observability.silver.{tabela}")

# 3. Gold e observability
for tabela in todas_as_tabelas_gold:
    spark.sql(f"DROP TABLE IF EXISTS poc_pulse_observability.gold.{tabela}")
    spark.sql(f"DROP TABLE IF EXISTS poc_pulse_observability.observability.{tabela}")

# 4. pipeline_runs
spark.sql("DROP TABLE IF EXISTS poc_pulse_observability.observability.pipeline_runs")

# 5. Landing Zone inteira (todos os sistemas)
for sistema in sistemas:
    dbutils.fs.rm(f"/Volumes/poc_pulse_observability/landing/raw/{sistema}", recurse=True)

print("Reset completo concluído.")

In [0]:
%pip install dbldatagen Faker

In [0]:
dbutils.library.restartPython()

## Piloto — 5 dias

Testa o loop de geração com uma janela pequena antes de escalar para os 60 dias completos.

In [0]:
from datetime import date, timedelta
from src.simuladores.simulador_factory import SimuladorFactory

data_inicio_piloto = date(2026, 8, 10)
data_fim_piloto = date(2026, 8, 14)

dia_atual = data_inicio_piloto
while dia_atual <= data_fim_piloto:
    print(f"--- Gerando {dia_atual} ---")
    for nome_sistema in SimuladorFactory.ordem_execucao():
        simulador = SimuladorFactory.criar(nome_sistema, spark=spark, dbutils=dbutils)
        simulador.executar_seed()
        resultado = simulador.gerar_dia(dia_atual)
        print(f"  {resultado['sistema']}: {resultado['status']} — {resultado.get('tabelas_geradas', [])}")
    dia_atual += timedelta(days=1)

print("\nPiloto concluído.")

## Backfill completo — 60 dias

Gera o histórico de 16/06/2026 a 14/08/2026, respeitando o calendário de cada sistema. Estimativa de ~14 minutos, baseada no piloto medido.

In [0]:
data_inicio = date(2026, 6, 16)
data_fim = date(2026, 8, 14)

inicio_execucao = time.time()
dia_atual = data_inicio
total_dias = 0

while dia_atual <= data_fim:
    for nome_sistema in SimuladorFactory.ordem_execucao():
        simulador = SimuladorFactory.criar(nome_sistema, spark=spark, dbutils=dbutils)
        simulador.executar_seed()
        resultado = simulador.gerar_dia(dia_atual)
    total_dias += 1
    if total_dias % 10 == 0:
        print(f"Progresso: {total_dias} dias processados ({dia_atual})")
    dia_atual += timedelta(days=1)

duracao_total = time.time() - inicio_execucao
print(f"\nBackfill concluído: {total_dias} dias em {duracao_total/60:.1f} minutos")

## Ingestão — Landing → Bronze (única execução, todos os 60 dias)

O Autoloader processa todo o histórico acumulado de uma vez — já validado que soma corretamente múltiplos micro-lotes numa única chamada (Lição 10).

In [0]:
from src.ingestao.ingestor_autoloader import IngestorAutoloader

TABELAS_POR_SISTEMA = {
    "erp": ["erp_lotes_producao", "erp_posicoes_estoque", "erp_notas_expedicao"],
    "crm": ["crm_pedidos", "crm_itens_pedido", "crm_atendimento"],
    "tms": ["tms_remessas", "tms_leituras_temperatura", "tms_comprovantes_entrega"],
    "financeiro": ["financeiro_faturas", "financeiro_contas_receber"],
}

for sistema, tabelas in TABELAS_POR_SISTEMA.items():
    for tabela in tabelas:
        ingestor = IngestorAutoloader(spark=spark, sistema=sistema, tabela=tabela)
        resultado = ingestor.executar()
        print(resultado)

In [0]:
from src.transformacao.configuracao_tabelas import CONFIGURACAO_TABELAS
from src.transformacao.transformar_bronze_para_silver import transformar_bronze_para_silver

for tabela, config in CONFIGURACAO_TABELAS.items():
    resultado = transformar_bronze_para_silver(
        spark=spark, catalog="poc_pulse_observability", tabela=tabela, config=config
    )
    print(resultado)

In [0]:
from src.transformacao.configuracao_seeds import CONFIGURACAO_SEEDS
from src.transformacao.promover_seed import promover_seed

for tabela, config in CONFIGURACAO_SEEDS.items():
    resultado = promover_seed(spark=spark, tabela=tabela, config=config)
    print(resultado)

In [0]:
from pyspark.sql.functions import col, abs as spark_abs, when, count, sum as spark_sum, lit

# gold_reconciliacao_financeira
df_pedidos = spark.table("poc_pulse_observability.silver.crm_pedidos")
df_faturas = spark.table("poc_pulse_observability.silver.financeiro_faturas")
df_reconciliacao = (
    df_faturas.join(df_pedidos.select("pedido_id", "valor_total"), "pedido_id")
    .withColumn("divergencia_valor", col("valor_faturado") - col("valor_total"))
    .withColumn("divergente", spark_abs(col("divergencia_valor")) > 0.01)
    .select("fatura_id", "pedido_id", "valor_total", "valor_faturado", "divergencia_valor", "divergente", "data_faturamento")
)
df_reconciliacao.write.format("delta").mode("overwrite").saveAsTable("poc_pulse_observability.gold.gold_reconciliacao_financeira")
total = df_reconciliacao.count()
divergentes = df_reconciliacao.filter(col("divergente")).count()
print(f"gold_reconciliacao_financeira — Total: {total} | Divergentes: {divergentes} ({divergentes/total:.1%})")

# gold_otif
df_remessas = spark.table("poc_pulse_observability.silver.tms_remessas")
df_comprovantes = spark.table("poc_pulse_observability.silver.tms_comprovantes_entrega")
df_otif = (
    df_comprovantes.join(df_remessas.select("remessa_id", "data_entrega_prevista"), "remessa_id")
    .withColumn("no_prazo", when(col("pod_confirmado") == True, col("data_entrega_real") <= col("data_entrega_prevista")).otherwise(None))
    .select("remessa_id", "comprovante_id", "data_entrega_prevista", "data_entrega_real", "pod_confirmado", "no_prazo", "status_entrega")
)
df_otif.write.format("delta").mode("overwrite").saveAsTable("poc_pulse_observability.gold.gold_otif")
total = df_otif.count()
confirmadas = df_otif.filter(col("pod_confirmado") == True).count()
no_prazo = df_otif.filter(col("no_prazo") == True).count()
print(f"gold_otif — Total: {total} | Confirmadas: {confirmadas} | No prazo: {no_prazo} ({no_prazo/confirmadas:.1%})")

# gold_qualidade_producao
df_lotes = spark.table("poc_pulse_observability.silver.erp_lotes_producao")
df_qualidade = (
    df_lotes.groupBy("centro_producao_id", "produto_id")
    .agg(count("*").alias("total_lotes"), spark_sum(when(col("status_qc") == "reprovado", 1).otherwise(0)).alias("lotes_reprovados"))
    .withColumn("taxa_rejeicao", col("lotes_reprovados") / col("total_lotes"))
)
df_qualidade.write.format("delta").mode("overwrite").saveAsTable("poc_pulse_observability.gold.gold_qualidade_producao")
print(f"gold_qualidade_producao — {df_qualidade.count()} combinações centro+produto")

In [0]:
# observability_cadeia_fria
df_notas = spark.table("poc_pulse_observability.silver.erp_notas_expedicao").select("nota_expedicao_id", "lote_id")
df_lotes_fk = spark.table("poc_pulse_observability.silver.erp_lotes_producao").select("lote_id", "produto_id")
df_produtos = spark.table("poc_pulse_observability.silver.erp_produtos").select("produto_id", "nome_produto", "exige_cadeia_fria")
df_veiculos = spark.table("poc_pulse_observability.silver.tms_veiculos").select("veiculo_id", "refrigerado")

df_base = (
    df_remessas.select("remessa_id", "nota_expedicao_id", "veiculo_id")
    .join(df_notas, "nota_expedicao_id", "left")
    .join(df_lotes_fk, "lote_id", "left")
    .join(df_produtos, "produto_id", "left")
    .join(df_veiculos, "veiculo_id", "left")
    .withColumn("verificavel", col("produto_id").isNotNull() & col("refrigerado").isNotNull())
)

df_leituras = spark.table("poc_pulse_observability.silver.tms_leituras_temperatura")
df_temperatura_fora_faixa = (
    df_leituras.filter((col("temperatura_celsius") < 2.0) | (col("temperatura_celsius") > 8.0))
    .select("remessa_id").distinct().withColumn("teve_leitura_fora_faixa", lit(True))
)
df_com_temperatura = df_base.join(df_temperatura_fora_faixa, "remessa_id", "left").fillna({"teve_leitura_fora_faixa": False})

df_classificado = (
    df_com_temperatura.withColumn(
        "tipo_violacao",
        when(~col("verificavel"), "nao_verificavel")
        .when(~col("exige_cadeia_fria"), "nao_aplicavel")
        .when(~col("refrigerado"), "veiculo_incorreto")
        .when(col("teve_leitura_fora_faixa"), "falha_equipamento")
        .otherwise("conforme")
    )
    .select("remessa_id", "veiculo_id", "produto_id", "nome_produto", "exige_cadeia_fria", "refrigerado", "teve_leitura_fora_faixa", "tipo_violacao")
)
df_classificado.write.format("delta").mode("overwrite").saveAsTable("poc_pulse_observability.observability.observability_cadeia_fria")
df_classificado.groupBy("tipo_violacao").count().orderBy(col("count").desc()).show()

# observability_qualidade_sku
df_itens = spark.table("poc_pulse_observability.silver.crm_itens_pedido")
df_produtos_validos = spark.table("poc_pulse_observability.silver.erp_produtos").select(col("produto_id").alias("produto_id_valido"))
df_sku = (
    df_itens.join(df_produtos_validos, df_itens.produto_id == col("produto_id_valido"), "left")
    .withColumn("sku_valido", col("produto_id_valido").isNotNull())
    .select("item_pedido_id", "pedido_id", "produto_id", "sku_valido")
)
df_sku.write.format("delta").mode("overwrite").saveAsTable("poc_pulse_observability.observability.observability_qualidade_sku")
total = df_sku.count()
invalidos = df_sku.filter(~col("sku_valido")).count()
print(f"observability_qualidade_sku — Total: {total} | SKU inválido: {invalidos} ({invalidos/total:.1%})")

# observability_estoque_negativo
df_estoque = spark.table("poc_pulse_observability.silver.erp_posicoes_estoque")
df_estoque_negativo = df_estoque.select("posicao_id", "lote_id", "centro_distribuicao_id", "quantidade", "data_posicao").withColumn("estoque_invalido", col("quantidade") < 0)
df_estoque_negativo.write.format("delta").mode("overwrite").saveAsTable("poc_pulse_observability.observability.observability_estoque_negativo")
total = df_estoque_negativo.count()
negativos = df_estoque_negativo.filter(col("estoque_invalido")).count()
print(f"observability_estoque_negativo — Total: {total} | Negativo: {negativos} ({negativos/total:.1%})")